# Interactive AVO temporal-resolution comparison

This notebook discovers and compiles AVO parquet files for each station and temporal resolution:

- `instant`
- `hourly`
- `daily`
- `monthly`

It scans the common `gawkenyadata/level1` tree, including both `level1/nrb` and `level1/buc`.

The chart is interactive:

- select the station and variable with dropdowns;
- click legend entries to show or hide individual temporal resolutions;
- double-click a legend entry to isolate one series;
- use Plotly zoom, pan, hover, range slider, and image-export controls.

Traces are added in this order: **instant → hourly → daily → monthly**. Plotly draws later traces above earlier traces, so monthly data appear on top of daily data, and daily data appear above hourly and instant data.

In [1]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import re
from typing import Iterable

import polars as pl
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

## Configuration

Set `GAWKENYADATA_ROOT` to the root of the `gawkenyadata` repository.

`DEFAULT_STATION` is only the initial selection. All stations are discovered automatically.

Set `WRITE_COMPILED_PARQUETS = True` to save one concatenated parquet per cadence below:

```text
gawkenyadata/level2/avo_temporal_comparison/<station>/
```

In [2]:
GAWKENYADATA_ROOT = Path(
    "/product_data/data/pay/Kenya/git/gawkenyadata"
)

LEVEL1_ROOT = GAWKENYADATA_ROOT / "level1"
LEVEL2_OUTPUT_ROOT = (
    GAWKENYADATA_ROOT
    / "level2"
    / "avo_temporal_comparison"
)

DEFAULT_STATION = "roof"
DEFAULT_VARIABLE = "pm25_conc"

WRITE_COMPILED_PARQUETS = False

CADENCE_ORDER = ("instant", "hourly", "daily", "monthly")

CADENCE_STYLE = {
    "instant": {
        "color": "rgba(110, 110, 110, 0.45)",
        "width": 1.0,
        "marker_size": 3,
    },
    "hourly": {
        "color": "rgba(31, 119, 180, 0.75)",
        "width": 1.5,
        "marker_size": 4,
    },
    "daily": {
        "color": "rgba(255, 127, 14, 0.90)",
        "width": 2.5,
        "marker_size": 6,
    },
    "monthly": {
        "color": "rgba(214, 39, 40, 1.00)",
        "width": 4.0,
        "marker_size": 9,
    },
}

PREFERRED_VARIABLES = {
    "pm1": "PM1 [µg/m³]",
    "pm25_conc": "PM2.5 [µg/m³]",
    "pm10_conc": "PM10 [µg/m³]",
    "co2": "CO₂",
    "pr": "Pressure",
    "hm": "Relative humidity",
    "tp": "Temperature",
}

if not LEVEL1_ROOT.is_dir():
    raise NotADirectoryError(
        f"Level-1 root does not exist: {LEVEL1_ROOT}"
    )

print(f"Level-1 source root: {LEVEL1_ROOT}")
print(f"Optional level-2 output root: {LEVEL2_OUTPUT_ROOT}")

Level-1 source root: /product_data/data/pay/Kenya/git/gawkenyadata/level1
Optional level-2 output root: /product_data/data/pay/Kenya/git/gawkenyadata/level2/avo_temporal_comparison


## Discover available AVO stations and files

In [3]:
AVO_FILE_RE = re.compile(
    r"^avo-(?P<station>.+)-"
    r"(?P<cadence>instant|hourly|daily|monthly)"
    r"\.parquet$",
    flags=re.IGNORECASE,
)


def discover_avo_files(
    level1_root: Path,
) -> dict[str, dict[str, list[Path]]]:
    discovered: dict[
        str, dict[str, list[Path]]
    ] = defaultdict(lambda: defaultdict(list))

    for path in sorted(level1_root.rglob("avo-*.parquet")):
        if not path.is_file():
            continue

        match = AVO_FILE_RE.match(path.name)
        if match is None:
            continue

        station = match.group("station").lower()
        cadence = match.group("cadence").lower()
        discovered[station][cadence].append(path)

    return {
        station: {
            cadence: sorted(paths)
            for cadence, paths in cadence_map.items()
        }
        for station, cadence_map in sorted(
            discovered.items()
        )
    }


discovered_files = discover_avo_files(LEVEL1_ROOT)

if not discovered_files:
    raise FileNotFoundError(
        f"No files matching avo-<station>-<cadence>.parquet "
        f"were found below {LEVEL1_ROOT}"
    )

print("Discovered stations:")
for station, cadence_map in discovered_files.items():
    summary = ", ".join(
        f"{cadence}={len(cadence_map.get(cadence, []))}"
        for cadence in CADENCE_ORDER
        if cadence_map.get(cadence)
    )
    print(f"  {station}: {summary}")

Discovered stations:
  garden: instant=3, hourly=3, daily=3, monthly=14
  huduma: instant=3, hourly=3, daily=4, monthly=24
  mogogosiek: instant=4, hourly=4, daily=6, monthly=21
  roof: instant=23, hourly=23, daily=25, monthly=36


## Compilation helpers

In [4]:
def ensure_dtm(
    df: pl.DataFrame,
    source: Path,
) -> pl.DataFrame:
    if "dtm" not in df.columns:
        raise ValueError(
            f"{source}: missing required 'dtm' column"
        )

    dtype = df.schema["dtm"]
    if isinstance(dtype, pl.Datetime):
        if dtype.time_zone is None:
            return df.with_columns(
                pl.col("dtm")
                .dt.replace_time_zone("UTC")
                .dt.cast_time_unit("us")
            )
        return df.with_columns(
            pl.col("dtm")
            .dt.convert_time_zone("UTC")
            .dt.cast_time_unit("us")
        )

    text = (
        pl.col("dtm")
        .cast(pl.Utf8, strict=False)
        .str.strip_chars()
    )

    parsed = pl.coalesce(
        [
            text.str.strptime(
                pl.Datetime(
                    time_unit="us",
                    time_zone="UTC",
                ),
                "%Y-%m-%dT%H:%M:%S%.f%#z",
                strict=False,
            ),
            text.str.strptime(
                pl.Datetime(
                    time_unit="us",
                    time_zone="UTC",
                ),
                "%Y-%m-%d %H:%M:%S%.f%#z",
                strict=False,
            ),
            text.str.strptime(
                pl.Datetime(time_unit="us"),
                "%Y-%m-%dT%H:%M:%S%.f",
                strict=False,
            ).dt.replace_time_zone("UTC"),
            text.str.strptime(
                pl.Datetime(time_unit="us"),
                "%Y-%m-%d %H:%M:%S%.f",
                strict=False,
            ).dt.replace_time_zone("UTC"),
            text.str.strptime(
                pl.Datetime(time_unit="us"),
                "%Y-%m-%d %H:%M:%S",
                strict=False,
            ).dt.replace_time_zone("UTC"),
            text.str.strptime(
                pl.Date,
                "%Y-%m-%d",
                strict=False,
            )
            .cast(pl.Datetime(time_unit="us"))
            .dt.replace_time_zone("UTC"),
        ]
    ).alias("dtm")

    result = df.with_columns(parsed)
    if result["dtm"].null_count() == result.height:
        raise ValueError(
            f"{source}: no timestamps could be parsed"
        )

    return result.filter(pl.col("dtm").is_not_null())


def compile_cadence(
    paths: Iterable[Path],
    *,
    station: str,
    cadence: str,
    verbose: bool = True,
) -> pl.DataFrame:
    path_list = list(paths)
    if not path_list:
        return pl.DataFrame()

    frames: list[pl.DataFrame] = []

    for index, path in enumerate(path_list, start=1):
        if verbose:
            print(
                f"[{station}/{cadence}] "
                f"{index}/{len(path_list)}: {path}"
            )

        frame = ensure_dtm(
            pl.read_parquet(path),
            source=path,
        )
        frame = frame.with_columns(
            pl.lit(str(path)).alias("_source_file")
        )
        frames.append(frame)

    compiled = (
        pl.concat(frames, how="diagonal_relaxed")
        .sort("dtm")
        .unique(
            subset=["dtm"],
            keep="last",
            maintain_order=True,
        )
        .sort("dtm")
    )

    if verbose:
        first_dtm = compiled["dtm"].min()
        last_dtm = compiled["dtm"].max()
        print(
            f"[{station}/{cadence}] compiled "
            f"{compiled.height:,} rows, "
            f"{first_dtm} to {last_dtm}"
        )

    if WRITE_COMPILED_PARQUETS:
        output_dir = LEVEL2_OUTPUT_ROOT / station
        output_dir.mkdir(
            parents=True,
            exist_ok=True,
        )
        output_path = (
            output_dir
            / f"avo-{station}-{cadence}-compiled.parquet"
        )
        compiled.drop("_source_file").write_parquet(
            output_path
        )
        print(f"Wrote: {output_path}")

    return compiled


_compiled_cache: dict[
    str, dict[str, pl.DataFrame]
] = {}


def compile_station(
    station: str,
    *,
    force: bool = False,
    verbose: bool = True,
) -> dict[str, pl.DataFrame]:
    station = station.lower()

    if station in _compiled_cache and not force:
        return _compiled_cache[station]

    if station not in discovered_files:
        raise KeyError(
            f"Unknown station: {station}"
        )

    compiled: dict[str, pl.DataFrame] = {}

    for cadence in CADENCE_ORDER:
        paths = discovered_files[station].get(
            cadence,
            [],
        )
        if not paths:
            continue

        compiled[cadence] = compile_cadence(
            paths,
            station=station,
            cadence=cadence,
            verbose=verbose,
        )

    _compiled_cache[station] = compiled
    return compiled


def numeric_variables(
    compiled: dict[str, pl.DataFrame],
) -> list[str]:
    variables: set[str] = set()
    excluded = {
        "dtm",
        "ts",
        "_source_file",
    }

    for df in compiled.values():
        for name, dtype in df.schema.items():
            if name in excluded:
                continue
            if dtype.is_numeric():
                variables.add(name)

    preferred = [
        variable
        for variable in PREFERRED_VARIABLES
        if variable in variables
    ]
    remaining = sorted(variables.difference(preferred))
    return preferred + remaining


def variable_label(variable: str) -> str:
    return PREFERRED_VARIABLES.get(
        variable,
        variable,
    )

## Interactive selector and plot

The first selection of a station compiles all its available cadence files. The result is cached in memory, so changing the variable does not reread the parquet files.

In [ ]:
available_stations = sorted(discovered_files)

initial_station = (
    DEFAULT_STATION
    if DEFAULT_STATION in available_stations
    else available_stations[0]
)

station_dropdown = widgets.Dropdown(
    options=available_stations,
    value=initial_station,
    description="Station:",
    layout=widgets.Layout(width="360px"),
    style={"description_width": "90px"},
)

variable_dropdown = widgets.Dropdown(
    options=[],
    description="Variable:",
    layout=widgets.Layout(width="440px"),
    style={"description_width": "90px"},
)

refresh_button = widgets.Button(
    description="Reload files",
    icon="refresh",
    tooltip="Clear the cache and reread this station",
)

status_output = widgets.Output()
plot_output = widgets.Output()


def refresh_variable_options(
    station: str,
    *,
    force: bool = False,
) -> dict[str, pl.DataFrame]:
    with status_output:
        clear_output(wait=True)
        print(f"Compiling station: {station}")
        compiled = compile_station(
            station,
            force=force,
            verbose=True,
        )

        variables = numeric_variables(compiled)
        if not variables:
            raise ValueError(
                f"No numeric variables found for {station}"
            )

        old_value = variable_dropdown.value
        variable_dropdown.options = [
            (variable_label(name), name)
            for name in variables
        ]

        if old_value in variables:
            variable_dropdown.value = old_value
        elif DEFAULT_VARIABLE in variables:
            variable_dropdown.value = DEFAULT_VARIABLE
        else:
            variable_dropdown.value = variables[0]

        print(
            "Available cadences: "
            + ", ".join(compiled)
        )
        print(
            "Available variables: "
            + ", ".join(variables)
        )

    return compiled


def build_figure(
    station: str,
    variable: str,
) -> go.Figure:
    compiled = compile_station(
        station,
        verbose=False,
    )

    fig = go.Figure()
    plotted = 0

    for cadence in CADENCE_ORDER:
        df = compiled.get(cadence)
        if (
            df is None
            or df.is_empty()
            or variable not in df.columns
        ):
            continue

        plot_df = (
            df.select(["dtm", variable])
            .drop_nulls()
            .sort("dtm")
        )
        if plot_df.is_empty():
            continue

        style = CADENCE_STYLE[cadence]
        mode = (
            "lines"
            if cadence in {"instant", "hourly"}
            else "lines+markers"
        )

        fig.add_trace(
            go.Scattergl(
                x=plot_df["dtm"].to_list(),
                y=plot_df[variable].to_list(),
                mode=mode,
                name=cadence.capitalize(),
                line={
                    "color": style["color"],
                    "width": style["width"],
                },
                marker={
                    "color": style["color"],
                    "size": style["marker_size"],
                },
                connectgaps=False,
                hovertemplate=(
                    f"{cadence.capitalize()}<br>"
                    "%{x|%Y-%m-%d %H:%M UTC}<br>"
                    f"{variable_label(variable)}: "
                    "%{y:.3f}"
                    "<extra></extra>"
                ),
            )
        )
        plotted += 1

    if plotted == 0:
        raise ValueError(
            f"No non-null '{variable}' data found "
            f"for station '{station}'."
        )

    fig.update_layout(
        title={
            "text": (
                f"AVO {station}: "
                f"{variable_label(variable)}"
            ),
            "font": {"size": 24},
        },
        xaxis={
            "title": {
                "text": "Date/time [UTC]",
                "font": {"size": 18},
            },
            "tickfont": {"size": 14},
            "rangeslider": {"visible": True},
        },
        yaxis={
            "title": {
                "text": variable_label(variable),
                "font": {"size": 18},
            },
            "tickfont": {"size": 14},
        },
        legend={
            "title": {
                "text": "Temporal resolution"
            },
            "font": {"size": 15},
            "itemclick": "toggle",
            "itemdoubleclick": "toggleothers",
        },
        hovermode="x unified",
        template="plotly_white",
        height=720,
        margin={
            "l": 90,
            "r": 40,
            "t": 90,
            "b": 80,
        },
    )

    return fig


def render_plot(*_args) -> None:
    station = station_dropdown.value
    variable = variable_dropdown.value

    if not station or not variable:
        return

    with plot_output:
        clear_output(wait=True)
        figure = build_figure(
            station=station,
            variable=variable,
        )
        figure.show(
            config={
                "displaylogo": False,
                "responsive": True,
                "scrollZoom": True,
                "toImageButtonOptions": {
                    "format": "png",
                    "filename": (
                        f"avo-{station}-{variable}"
                    ),
                    "scale": 2,
                },
            }
        )


def on_station_change(change) -> None:
    if change.get("name") != "value":
        return
    refresh_variable_options(
        change["new"],
        force=False,
    )
    render_plot()


def on_variable_change(change) -> None:
    if change.get("name") == "value":
        render_plot()


def on_refresh_clicked(_button) -> None:
    station = station_dropdown.value
    _compiled_cache.pop(station, None)
    refresh_variable_options(
        station,
        force=True,
    )
    render_plot()


station_dropdown.observe(
    on_station_change,
    names="value",
)
variable_dropdown.observe(
    on_variable_change,
    names="value",
)
refresh_button.on_click(on_refresh_clicked)

controls = widgets.HBox(
    [
        station_dropdown,
        variable_dropdown,
        refresh_button,
    ]
)

display(controls)
display(status_output)
display(plot_output)

refresh_variable_options(initial_station)
render_plot()

Output()

Output()

## Optional direct use without widgets

In [6]:
# Example:
#
# station = "huduma"
# variable = "pm25_conc"
#
# compiled = compile_station(station)
# figure = build_figure(station, variable)
# figure.show()
#
# figure.write_html(
#     GAWKENYADATA_ROOT
#     / "level2"
#     / f"avo-{station}-{variable}.html",
#     include_plotlyjs="cdn",
# )